# 🔬 Lab 03: Production Vector Stores & Metadata Payload Filtering

---

## 🎯 Executive Overview & Production Challenge
Moving from a toy vector database (Chroma in-memory) to a production-grade enterprise vector store (Qdrant / OpenSearch) requires:
1. **Strict Payload Schemata:** Indexing payload fields (facility, equipment type, severity, date ranges) for fast filtering.
2. **HNSW Vector Graph Configuration:** Tuning graph connectivity parameters ($M$, $ef\_construction$, $ef\_search$) for sub-5ms search.
3. **Hybrid Sparse + Dense Mechanics:** Combining dense semantic vectors with inverted keyword token indexes.
4. **Pre-Filtering vs Post-Filtering:** Ensuring the vector engine filters points **before** traversing the HNSW graph to avoid dropping recall on sparse subsets.

```
Incoming Query + Metadata Filter: { facility: 'Fahud', severity: 'CRITICAL' }
              │
              ▼
   [ Payload Inverted Index ] ──> Fast Bitset Filter (100 matches out of 1M)
              │
              ▼
   [ HNSW Graph Traversal ]   ──> Traverses ONLY the active bitset
              │
              ▼
   Top-K High-Precision Chunks Returned in < 4ms
```


In [ ]:
import json
import numpy as np
import tempfile
import os
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue, Range
)
from sentence_transformers import SentenceTransformer

# Configure sandboxed cache directories for Torch, HuggingFace, and Matplotlib
temp_dir = tempfile.gettempdir()
os.environ.setdefault("MPLCONFIGDIR", os.path.join(temp_dir, "matplotlib"))
os.environ.setdefault("TORCH_HOME", os.path.join(temp_dir, "torch"))
os.environ.setdefault("HF_HOME", os.path.join(temp_dir, "hf"))
os.environ.setdefault("TORCHINDUCTOR_CACHE_DIR", os.path.join(temp_dir, "torch_inductor"))

# Initialize local in-memory Qdrant instance
client = QdrantClient(":memory:")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
model = embedder  # Alias for backward compatibility
print("[✓] Initialized embedded in-memory Qdrant client and SentenceTransformer model.")


---
## Step 1: Collection Creation & HNSW Graph Tuning
We define a collection with 384-dimensional vectors, Cosine distance metric, and explicitly configured HNSW parameters.


In [ ]:
COLLECTION_NAME = "ust_operations_vault"

client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=384, distance=Distance.COSINE)
)
print(f"[✓] Created Qdrant collection '{COLLECTION_NAME}' (dim=384, distance=COSINE).")


---
## Step 2: Indexing Payload Fields
In production, searching without indexing payload fields results in slow sequential table scans. We create payload indexes on `facility` and `severity`.


In [ ]:
from qdrant_client.models import PayloadSchemaType

# Index facility keyword
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="facility",
    field_schema=PayloadSchemaType.KEYWORD
)

# Index severity keyword
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="severity",
    field_schema=PayloadSchemaType.KEYWORD
)

# Index downtime hours as float range
client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="downtime_hours",
    field_schema=PayloadSchemaType.FLOAT
)
print("[✓] Created production payload indexes on: facility, severity, downtime_hours.")


---
## Step 3: Ingestion of Dense Vectors with Rich Payloads
We load our structured incident dataset, compute embeddings using local `SentenceTransformer`, and upsert points into Qdrant.


In [ ]:
import sys

# Self-healing sys.path resolution so rag_universal_client is ALWAYS found
for candidate in [
    Path.cwd(),
    Path.cwd() / "RAG_ContxtEngg_Labs",
    Path(__file__).parent if "__file__" in locals() else None,
    Path.cwd() / "UST" / "RAG_ContxtEngg_Labs",
    Path(r"c:\Users\jadid\Desktop\MyWork\Techademy\UST\RAG_ContxtEngg_Labs")
]:
    if candidate and candidate.exists() and (candidate / "rag_universal_client.py").exists():
        if str(candidate.resolve()) not in sys.path:
            sys.path.insert(0, str(candidate.resolve()))
        break
import rag_universal_client as rc

# Resilient path resolution for sample_documents
DATA_DIR = rc.get_rag_data_dir()
print(f"[*] Verified Data Directory: {DATA_DIR}")

# Load expanded 25+ incident reports dataset
with open(DATA_DIR / "incident_reports.json", "r", encoding="utf-8") as f:
    incidents = json.load(f)

print(f"[✓] Loaded {len(incidents)} industrial incident reports.")

# Generate dense vector embeddings & payloads
points = []
for idx, inc in enumerate(incidents):
    # Dense contextual string for semantic search
    doc_text = (
        f"Incident {inc['incident_id']} at {inc['facility']} ({inc['equipment_tag']}). "
        f"Subsystem: {inc['subsystem']} | Severity: {inc['severity']} | Downtime: {inc['downtime_hours']} hrs. "
        f"Event: {inc['event_description']} "
        f"Root cause: {inc['root_cause_analysis']} "
        f"Corrective action: {inc['corrective_action']}"
    )
    vec = embedder.encode(doc_text, normalize_embeddings=True).tolist()
    
    # Store complete metadata payload for high-performance pre-filtering
    payload = {
        "incident_id": inc["incident_id"],
        "facility": inc["facility"],
        "equipment_tag": inc["equipment_tag"],
        "equipment_type": inc.get("equipment_type", "Turbomachinery"),
        "subsystem": inc["subsystem"],
        "severity": inc["severity"],
        "downtime_hours": float(inc["downtime_hours"]),
        "financial_impact_usd": float(inc.get("financial_impact_usd", 0.0)),
        "assigned_engineer": inc.get("assigned_engineer", "Unassigned"),
        "text": doc_text,
        "full_text": doc_text  # Cross-compatible key
    }
    
    points.append(PointStruct(id=idx + 1, vector=vec, payload=payload))

# Batch upsert to Qdrant collection
client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"[✓] Successfully ingested {len(points)} vector points into Qdrant collection '{COLLECTION_NAME}'.")


---
## Step 4: Strict Payload Filtering (Boolean Conjunction & Range Queries)
Now let's execute a semantic search restricted strictly to:
- `facility`: "Fahud Central Compression Station"
- `severity`: "CRITICAL"
- `downtime_hours`: $\ge 5.0$


In [ ]:
query_text = "Gas leakage or seal element failure"
query_vector = embedder.encode(query_text, normalize_embeddings=True).tolist()

search_filter = Filter(
    must=[
        FieldCondition(key="facility", match=MatchValue(value="Fahud Central Processing Facility")),
        FieldCondition(key="severity", match=MatchValue(value="CRITICAL")),
        FieldCondition(key="downtime_hours", range=Range(gte=5.0))
    ]
)

# Compatible search supporting both qdrant-client >=1.10 (query_points) and older (<1.10)
if hasattr(client, "query_points"):
    res = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        query_filter=search_filter,
        limit=3
    )
    filtered_results = res.points
else:
    filtered_results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=query_vector,
        query_filter=search_filter,
        limit=3
    )

print(f"Query: '{query_text}' with Strict Pre-Filter:")
for hit in filtered_results:
    print(f"\n[Match ID {hit.id}] Score: {hit.score:.4f}")
    print(f"  Facility: {hit.payload['facility']} | Equipment: {hit.payload['equipment_tag']}")
    print(f"  Severity: {hit.payload['severity']} | Downtime: {hit.payload['downtime_hours']} hrs")
    print(f"  Summary: {hit.payload['full_text']}")


---
## 🏁 Production Engineering Takeaways & Exercises
1. **Always build payload indexes** on high-cardinality metadata fields (e.g. `tenant_id`, `facility`, `category`).
2. **Qdrant uses Single-Stage HNSW Filtering:** It filters candidate bitsets *during* graph traversal, avoiding the recall drop associated with naive post-filtering.
3. **Docker Deployment Command for Production:**
   ```bash
   docker run -p 6333:6333 -v $(pwd)/qdrant_storage:/qdrant/storage:z qdrant/qdrant
   ```
